# StableSteering — Session Analysis Template

This notebook loads the tidy CSV export produced by `scripts/export_session_csv.py`
and generates standard analysis plots:

1. Steering vector trajectory (z magnitude over rounds)
2. Tag frequency heatmap (critique_rating sessions)
3. Candidate diversity per round
4. Updater comparison (if multiple sessions loaded)

**Usage:**
```bash
python scripts/export_session_csv.py <session_id>
# then open this notebook and set SESSION_DIR below
```

In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
SESSION_DIR = Path("../output/exports")  # change to your session export path
# e.g. Path("../output/exports/ses_abc123")

# Find the first session dir if SESSION_DIR points to the exports root
if not (SESSION_DIR / "rounds.csv").exists():
    candidates = sorted(SESSION_DIR.glob("ses_*"))
    if candidates:
        SESSION_DIR = candidates[0]
        print(f"Using session: {SESSION_DIR.name}")
    else:
        print("No session exports found. Run: python scripts/export_session_csv.py <session_id>")

rounds_df = pd.read_csv(SESSION_DIR / "rounds.csv")
candidates_df = pd.read_csv(SESSION_DIR / "candidates.csv")
feedback_df = pd.read_csv(SESSION_DIR / "feedback.csv")

print(f"Rounds: {len(rounds_df)}  Candidates: {len(candidates_df)}  Feedback events: {len(feedback_df)}")
rounds_df.head()

## 1. Steering Vector Trajectory

In [ ]:
# Compute z magnitude per round from candidates (winner = first by round_index)
def z_magnitude(z_json: str) -> float:
    z = json.loads(z_json)
    return math.sqrt(sum(v**2 for v in z))

# Use the incumbent candidate per round
winner_ids = rounds_df[["round_index", "winner_candidate_id"]]
merged = candidates_df.merge(winner_ids, on="round_index", how="left")
incumbents = merged[merged["candidate_id"] == merged["winner_candidate_id"]].copy()
incumbents["z_magnitude"] = incumbents["z"].apply(z_magnitude)

plt.figure(figsize=(8, 4))
plt.plot(incumbents["round_index"], incumbents["z_magnitude"], marker="o", linewidth=2)
plt.xlabel("Round")
plt.ylabel("‖z‖ (steering vector magnitude)")
plt.title("Steering Vector Trajectory")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Tag Frequency Heatmap (critique_rating sessions)

In [ ]:
from collections import Counter

tag_counts: Counter = Counter()
for tags_json in feedback_df["critique_tags_json"].dropna():
    tags_dict = json.loads(tags_json)
    for tag_list in tags_dict.values():
        tag_counts.update(tag_list)

if tag_counts:
    tags, counts = zip(*sorted(tag_counts.items(), key=lambda x: -x[1]))
    colors = ["#4caf50" if t in {"good_composition","good_color","good_detail"} else "#f44336" for t in tags]
    plt.figure(figsize=(10, 4))
    plt.bar(tags, counts, color=colors)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Times selected")
    plt.title("Critique Tag Frequency (green = positive, red = negative)")
    plt.tight_layout()
    plt.show()
else:
    print("No critique tags found (non-critique session)")

## 3. Candidate Diversity Per Round

In [ ]:
def mean_pairwise_distance(group):
    zs = [json.loads(z) for z in group["z"]]
    if len(zs) < 2:
        return 0.0
    pairs = [(zs[i], zs[j]) for i in range(len(zs)) for j in range(i+1, len(zs))]
    return sum(math.sqrt(sum((a-b)**2 for a,b in zip(x,y))) for x,y in pairs) / len(pairs)

diversity = candidates_df.groupby("round_index").apply(mean_pairwise_distance).reset_index()
diversity.columns = ["round_index", "mean_pairwise_dist"]

plt.figure(figsize=(8, 4))
plt.plot(diversity["round_index"], diversity["mean_pairwise_dist"], marker="s", color="purple", linewidth=2)
plt.xlabel("Round")
plt.ylabel("Mean pairwise distance")
plt.title("Candidate Diversity Per Round")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Updater Comparison (multi-session)

In [ ]:
# Load all sessions from the exports root for cross-session comparison
exports_root = SESSION_DIR.parent
all_rounds = []
for session_path in sorted(exports_root.glob("ses_*")):
    rdf = pd.read_csv(session_path / "rounds.csv")
    all_rounds.append(rdf)

if len(all_rounds) > 1:
    combined = pd.concat(all_rounds, ignore_index=True)
    summary = combined.groupby(["session_id", "updater"]).agg(
        rounds=("round_index", "count"),
        converged=("converged", "first"),
    ).reset_index()
    print(summary.to_string(index=False))
else:
    print("Only one session found. Export more sessions to compare updaters.")